# Delta Lake + Unity Catalog + MinIO Quickstart
This notebook initializes a Spark session configured for Delta Lake and Unity Catalog, and connects to your local MinIO server.

In [1]:
from pyspark.sql import SparkSession

print("Starting Spark Session... This might take a minute the first time to download jars.")
spark = SparkSession.builder \
    .appName("DataCore_Delta") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0,io.unitycatalog:unitycatalog-spark_2.12:0.2.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "io.unitycatalog.spark.UCSingleCatalog") \
    .config("spark.sql.catalog.unity", "io.unitycatalog.spark.UCSingleCatalog") \
    .config("spark.sql.catalog.unity.uri", "http://unitycatalog:8080") \
    .config("spark.sql.defaultCatalog", "unity") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()

print("Spark Session Created Successfully!")

Starting Spark Session... This might take a minute the first time to download jars.
Spark Session Created Successfully!


In [3]:
# Create a test dataframe
data = [("Ahmed", 25), ("Omar", 30), ("Sara", 22)]
df = spark.createDataFrame(data, ["Name", "Age"])

df.show()

# Write the dataframe to MinIO as a Delta table
# IMPORTANT: Make sure you created a bucket named 'bronze' in MinIO first!
s3_path = "s3a://bronze/test_delta_table"
df.write.format("delta").mode("overwrite").save(s3_path)
print(f"Data written to {s3_path} successfully as a Delta Table!")

+-----+---+
| Name|Age|
+-----+---+
|Ahmed| 25|
| Omar| 30|
| Sara| 22|
+-----+---+

Data written to s3a://bronze/test_delta_table successfully as a Delta Table!
